Uus tabel, kus on:

pat_id (patterns.id)

head_id (transaction.head_id)

transaction_id (transaction.id)

phrase_nr (pattern.phrase_nr)

root (transaction.lemma)

Uue tabeli loomiseks:

1. transaction_head.verb matchib patterns.verb
2. transaction_head.id matchib transactions.head_id  
3. patterns.deprel + kääne peab matchima transaction.deptrel+kääne 


In [1]:
import sqlite3

In [2]:
con = sqlite3.connect("vp_data2_isikud.db")
cur = con.cursor()

In [3]:
# transaktsioonide andmebaasi lisamine (v33)
cur.execute('ATTACH DATABASE "v33.db" AS v33')

In [4]:
STAT = 'vahel'

### Luua uus tabel pat_tr_head_v1, kus on:

head_id, pat_id, verb_word, phrase_nr, phrase_case, deprel (verbi deprel)

Tabeli loomine:

Teha join transaction_head tabeliga verbi alusel.


In [5]:
cur.execute("""
DROP TABLE IF EXISTS pat_tr_head_{stat}
""".format(stat=STAT))

In [6]:
%%time

cur.execute("""
CREATE TABLE pat_tr_head_{stat} AS
SELECT DISTINCT
    head.id as head_id,
    pat.pat_id as pat_id,
    pat.verb_word as verb_word,
    pat.phrase_nr as phrase_nr,
    pat.phrase_case as phrase_case,
    pat.pat_deprel as pat_deprel
FROM 
    patterns_isikud_{stat2} as pat
INNER JOIN 
    transaction_head as head
ON
    pat.verb_word = head.verb

""".format(stat=STAT, stat2=STAT))

CPU times: user 2min 47s, sys: 1min 25s, total: 4min 12s
Wall time: 23min 12s


### Luua uus tabel patterns_transaction_isikud_*, kus on:

head_id, pat_id, transaction_id, phrase_nr, verb_word, root_word, verb_deprel, word_deprel, pos, phrase_case

Join toimub pat_tr_head_v1 ja transaction vahel. 

Praegu on joini aluseks head_id, deprel ja feats. 

` NB!!! patterns tabeli kääne on abl/all/ad jne ja neid tulebks matchida feats veerus olevaga`

In [9]:
%%time 

cur.execute("""
DROP TABLE IF EXISTS patterns_transaction_isikud_{stat}
""".format(stat=STAT))

cur.execute("""
CREATE TABLE patterns_transaction_isikud_{stat} AS
SELECT DISTINCT
    tbl1.head_id as head_id,
    tbl1.pat_id as pat_id,
    tr.id as transaction_id,
    tbl1.phrase_nr as phrase_nr,
    tbl1.verb_word as verb_word,
    tr.lemma as root_word,
    tbl1.pat_deprel as pat_deprel,
    tr.deprel as word_deprel,
    tr.pos as pos,
    tbl1.phrase_case as phrase_case,
    tr.feats as tr_feats
    
FROM 
    pat_tr_head_{stat2} as tbl1
JOIN 
    'transaction' as tr
ON 
    tbl1.head_id = tr.head_id
    and tbl1.pat_deprel = tr.deprel
WHERE INSTR(',' || tr.feats || ',', ',' || tbl1.phrase_case || ',') > 0

""".format(stat=STAT, stat2=STAT))

CPU times: user 2min, sys: 18.2 s, total: 2min 18s
Wall time: 2min 44s


### sama tabel, mis eelmine, juures on koht ja elus veerud

In [13]:
%%time 

cur.execute("""
DROP TABLE IF EXISTS patterns_transaction_isikud_{stat}_2
""".format(stat=STAT))

cur.execute("""
CREATE TABLE patterns_transaction_isikud_{stat}_2 AS
SELECT DISTINCT
    tbl1.head_id as head_id,
    tbl1.pat_id as pat_id,
    tr.id as transaction_id,
    tbl1.phrase_nr as phrase_nr,
    tbl1.verb_word as verb_word,
    tr.lemma as root_word,
    tbl1.pat_deprel as pat_deprel,
    tr.deprel as word_deprel,
    tr.pos as pos,
    tbl1.phrase_case as phrase_case,
    tr.feats as tr_feats,
    tr.koht as koht,
    tr.elus as elus
    
FROM 
    pat_tr_head_{stat2} as tbl1
JOIN 
    transaction_v2 as tr
ON 
    tbl1.head_id = tr.head_id
    and tbl1.pat_deprel = tr.deprel
WHERE INSTR(',' || tr.feats || ',', ',' || tbl1.phrase_case || ',') > 0

""".format(stat=STAT, stat2=STAT))

CPU times: user 2min 12s, sys: 25.2 s, total: 2min 37s
Wall time: 2min 47s


In [16]:
con.close()